# Day 5 · 从零手搭 Mini-VLM

**配套讲义**: `days/day-05.md` ｜ **需要 GPU**（torch + transformers）

今天的目标：把 SigLIP + Connector + Qwen2.5-0.5B 拼成 MiniVLM，
并**亲眼看到**视觉 embedding 如何替换 `<image>` 占位符。

## 0. 环境检查

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu  :", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.0f} GB")

## 1. 三块组件各自的输出形状

先分别跑一遍，记住每个环节的 shape —— 后面拼装出问题时按这个对照。

In [ ]:
import sys; sys.path.insert(0, "..")
import torch
from src.minivlm.vision import SiglipVisionWrapper
from src.minivlm.connector import build_connector

device = "cuda" if torch.cuda.is_available() else "cpu"

vision = SiglipVisionWrapper().to(device).eval()
conn = build_connector("mlp2",
                       vis_dim=1152, llm_dim=896).to(device).eval()

from PIL import Image, ImageDraw
img = Image.new("RGB", (448, 448), "white")
d = ImageDraw.Draw(img); d.ellipse([80, 80, 360, 360], fill=(230, 140, 60))

with torch.no_grad():
    vis = vision([img])            # [1, N_patches, 1152]
    proj = conn(vis)               # [1, N_patches, 896]
print("vision  :", tuple(vis.shape))
print("connector:", tuple(proj.shape))

## 2. 拼装 MiniVLM 并观察 merge

In [ ]:
from src.minivlm.model import MiniVLM, MiniVLMConfig
from transformers import AutoTokenizer

cfg = MiniVLMConfig()
model = MiniVLM(cfg).to(device).eval()
tok = AutoTokenizer.from_pretrained(cfg.llm_name)

text = "<image>这张图里有什么？"
ids = tok(text, return_tensors="pt").input_ids.to(device)
n_img = (ids == model.image_token_id).sum().item()
print(f"文本 token: {ids.shape[1]}，其中占位符: {n_img}")

with torch.no_grad():
    out = model(input_ids=ids, images=[img])
print("merged 序列长度:", out.logits.shape[1])
print("期望值 =", ids.shape[1] - n_img + proj.shape[1], "（文本-占位符+视觉）")

## 3. 反例实验：视觉 embedding 换成全零

形状没变、信息归零。如果 loss 还能算 —— 说明形状正确但「语义」才是关键。

In [ ]:
with torch.no_grad():
    vis_zero = torch.zeros_like(vision([img]))
    proj_zero = conn(vis_zero)
    out_zero = model(input_ids=ids, images=[img], visual_override=proj_zero) \
        if hasattr(model, "visual_override") else model(input_ids=ids, images=[img])
print("正常输出 logits 均值:", out.logits.mean().item())
print("零视觉  logits 均值:", out_zero.logits.mean().item())
print("两个值不同 → 视觉信息真的参与了计算")

## 4. 今日验收

- [ ] merged 长度 = 文本 - 占位符 + 视觉 token
- [ ] 手画数据流存 `assets/day5-flow.png`
- [ ] 回答：LLM 看到的到底是什么？

**卡住了？** 回看 `days/day-05.md` 第五节「容易踩的坑」（dtype / mask / 多占位符）。